# Gray-Box AIA72 — Mondrian Operational Policy v2 Freeze

Freezes the v2 NORMAL / DEGRADED / ABSTAIN policy using class-conditional Mondrian conformal states and the already-frozen q80/q90/q95 uncertainty gates. **No future outcomes are evaluated in this notebook.**

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd

# ============================================================
# Gray-Box AIA72 — Freeze operational policy v2
# Mondrian/class-conditional conformal + pre-frozen q80/q90/q95 gates
# ============================================================
#
# This notebook does NOT evaluate future outcomes.
# It reuses:
#   - raw/raw/raw probability calibration
#   - Mondrian v2 class-conditional conformal thresholds
#   - the already-frozen q80/q90/q95 uncertainty/applicability cut-offs
#   - the already-predeclared q90 primary policy
#
# Importantly:
#   - policy_validation results are NOT used to fit or select this v2 policy.
#   - q90 remains primary because it was predeclared before policy-validation.
#   - retrospective_cycle25 and supplementary_2026 outcomes remain unopened here.

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

MOND = Path("/home/abmoses2000/graybox_aia72_conformal_mondrian_v2_20261005")
MOND_PROTOCOL = MOND / "protocol.json"
MOND_THRESHOLDS = MOND / "mondrian_thresholds.csv"

POLICY_V1 = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005")
CUTS = POLICY_V1 / "candidate_cutoffs.csv"

ALARM_V1 = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
ALARM_PROTOCOL = ALARM_V1 / "protocol.json"
PRIMARY_DECL = ALARM_V1 / "primary_policy_declaration.json"

OUT = Path("/home/abmoses2000/graybox_aia72_policy_mondrian_v2_freeze_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

ROLE = "conformal_calibration"
EXPECTED_ROWS = 4168
PRIMARY_ALPHA = 0.10
SEEDS = [17,29,43]
PERCENTILES = [0.80,0.90,0.95]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond,msg):
    if not cond:
        raise ValueError(msg)

def entropy_binary(p):
    p=np.asarray(p,dtype=float)
    q=np.clip(p,1e-12,1-1e-12)
    return -(q*np.log(q)+(1-q)*np.log(1-q))

def mondrian_state(p,q0,q1):
    include0 = p <= q0
    include1 = (1.0-p) <= q1
    out=np.empty(len(p),dtype=object)
    out[(~include0)&(~include1)]="empty"
    out[include0&(~include1)]="singleton_no_flare"
    out[(~include0)&include1]="singleton_flare"
    out[include0&include1]="doubleton"
    return out

for p in [MASTER,SHARP,MOND_PROTOCOL,MOND_THRESHOLDS,CUTS,ALARM_PROTOCOL,PRIMARY_DECL]:
    require(p.is_file(), f"Missing frozen input: {p}")

require(sha256(MASTER)==MASTER_SHA,"Master SHA changed")
require(sha256(SHARP)==SHARP_SHA,"SHARP SHA changed")

mond_protocol=json.loads(MOND_PROTOCOL.read_text())
alarm_protocol=json.loads(ALARM_PROTOCOL.read_text())
primary_decl=json.loads(PRIMARY_DECL.read_text())

require(mond_protocol["status"]=="MONDRIAN_CLASS_CONDITIONAL_CONFORMAL_V2_FROZEN",
        "Mondrian v2 is not frozen")
require(mond_protocol["policy_validation_used_to_fit_v2"] is False,
        "Mondrian v2 used policy-validation for fitting")
require(mond_protocol["later_roles_read"] is False,
        "Mondrian v2 read later roles")
require(alarm_protocol["policy_validation_outcomes_read"] is False,
        "Alarm freeze was contaminated")
require(primary_decl["primary_policy_id"]=="policy_q90",
        "Primary v1 declaration changed")
require(primary_decl["selection_on_policy_validation_allowed"] is False,
        "Policy-validation selection unexpectedly allowed")

cuts=pd.read_csv(CUTS)
mond_thresh=pd.read_csv(MOND_THRESHOLDS)

# Read only the conformal-calibration role to reconstruct policy signals.
master=pd.read_csv(MASTER,low_memory=False)
sharp=pd.read_csv(SHARP,low_memory=False)
a=master.loc[master["role"].astype(str)==ROLE].copy()
s=sharp.loc[sharp["role"].astype(str)==ROLE].copy()

require(len(a)==EXPECTED_ROWS and len(s)==EXPECTED_ROWS,"Calibration support changed")
a["forecast_case_id"]=a["forecast_case_id"].astype(str)
s["forecast_case_id"]=s["forecast_case_id"].astype(str)
require(a["forecast_case_id"].tolist()==s["forecast_case_id"].tolist(),
        "SHARP/AIA calibration ID order mismatch")

aia_seed=a[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
sharp_seed=s[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
aia_p=pd.to_numeric(a["probability"],errors="raise").to_numpy(float)
sharp_p=pd.to_numeric(s["probability_gru_mean"],errors="raise").to_numpy(float)
fusion_seed=0.5*sharp_seed+0.5*aia_seed
fusion_p=0.5*sharp_p+0.5*aia_p

signals=pd.DataFrame({
    "forecast_case_id":a["forecast_case_id"].to_numpy(),
    "issue_utc":a["issue_utc"].to_numpy() if "issue_utc" in a.columns else pd.NA,
    "sharp_probability":sharp_p,
    "aia_probability":aia_p,
    "fusion_probability":fusion_p,
    "sharp_seed_std":sharp_seed.std(axis=1,ddof=1),
    "aia_seed_std":aia_seed.std(axis=1,ddof=1),
    "fusion_seed_std":fusion_seed.std(axis=1,ddof=1),
    "sharp_aia_probability_gap":np.abs(sharp_p-aia_p),
    "fusion_entropy":entropy_binary(fusion_p),
    "aia_input_status":a["input_status"].astype(str).to_numpy() if "input_status" in a.columns else "ok",
    "sharp_input_status":"ok",
})

q={}
for branch in ["sharp","aia","sharp_aia"]:
    hit=mond_thresh.loc[
        (mond_thresh["branch"]==branch) &
        np.isclose(mond_thresh["alpha"].astype(float),PRIMARY_ALPHA)
    ]
    require(len(hit)==1,f"Missing primary Mondrian threshold: {branch}")
    q[branch]={
        "q0":float(hit.iloc[0]["qhat_class0"]),
        "q1":float(hit.iloc[0]["qhat_class1"]),
    }

signals["sharp_mondrian_state"]=mondrian_state(sharp_p,q["sharp"]["q0"],q["sharp"]["q1"])
signals["aia_mondrian_state"]=mondrian_state(aia_p,q["aia"]["q0"],q["aia"]["q1"])
signals["fusion_mondrian_state"]=mondrian_state(fusion_p,q["sharp_aia"]["q0"],q["sharp_aia"]["q1"])

def cutoff(metric,pct):
    hit=cuts.loc[
        (cuts["metric"]==metric) &
        np.isclose(cuts["percentile"].astype(float),pct)
    ]
    require(len(hit)==1,f"Missing frozen cutoff {metric} q{pct}")
    return float(hit.iloc[0]["cutoff"])

candidate_policies=[]
state_counts={}

for pct in PERCENTILES:
    fs=cutoff("fusion_seed_std",pct)
    gap=cutoff("sharp_aia_probability_gap",pct)
    ent=cutoff("fusion_entropy",pct)
    ss=cutoff("sharp_seed_std",pct)

    fusion_single=np.isin(
        signals["fusion_mondrian_state"].to_numpy(),
        ["singleton_no_flare","singleton_flare"]
    )
    sharp_single=np.isin(
        signals["sharp_mondrian_state"].to_numpy(),
        ["singleton_no_flare","singleton_flare"]
    )

    aia_ok=(signals["aia_input_status"].to_numpy()=="ok")
    sharp_ok=(signals["sharp_input_status"].to_numpy()=="ok")

    normal=(
        aia_ok &
        fusion_single &
        (signals["fusion_seed_std"].to_numpy()<=fs) &
        (signals["sharp_aia_probability_gap"].to_numpy()<=gap) &
        (signals["fusion_entropy"].to_numpy()<=ent)
    )

    degraded=(
        (~normal) &
        sharp_ok &
        sharp_single &
        (signals["sharp_seed_std"].to_numpy()<=ss)
    )

    abstain=~(normal|degraded)

    require(np.all(normal.astype(int)+degraded.astype(int)+abstain.astype(int)==1),
            f"Non-exclusive states at q{pct}")

    pid=f"mondrian_v2_q{int(pct*100)}"
    state=np.where(normal,"NORMAL",np.where(degraded,"DEGRADED","ABSTAIN"))
    signals[f"{pid}_state"]=state

    state_counts[pid]={
        "NORMAL":int(normal.sum()),
        "DEGRADED":int(degraded.sum()),
        "ABSTAIN":int(abstain.sum()),
        "availability_rate":float((normal|degraded).mean()),
    }

    candidate_policies.append({
        "policy_id":pid,
        "cutoff_percentile":pct,
        "primary_conformal_alpha":PRIMARY_ALPHA,
        "normal_rule":{
            "all":[
                "aia_input_status == 'ok'",
                "fusion_mondrian_state in {'singleton_no_flare','singleton_flare'}",
                f"fusion_seed_std <= {fs:.17g}",
                f"sharp_aia_probability_gap <= {gap:.17g}",
                f"fusion_entropy <= {ent:.17g}",
            ],
            "forecast_source":"fixed 0.5/0.5 SHARP+AIA fusion",
        },
        "degraded_rule":{
            "all":[
                "NORMAL rule is false",
                "sharp_input_status == 'ok'",
                "sharp_mondrian_state in {'singleton_no_flare','singleton_flare'}",
                f"sharp_seed_std <= {ss:.17g}",
            ],
            "forecast_source":"SHARP fallback",
        },
        "abstain_rule":"Neither NORMAL nor DEGRADED rule is satisfied",
        "important_interpretation":[
            "A Mondrian doubleton is explicit class ambiguity and cannot be NORMAL.",
            "If fusion is doubleton or otherwise fails trust gates, SHARP may provide a DEGRADED fallback only when SHARP itself is a trusted singleton.",
            "Empty Mondrian sets, if any, cannot be NORMAL.",
            "No future outcomes were used to tune these rules.",
        ],
    })

policy_family={
    "status":"MONDRIAN_V2_POLICY_FAMILY_FROZEN",
    "scientific_acceptance":False,
    "primary_policy_id":"mondrian_v2_q90",
    "primary_inheritance_reason":(
        "q90 remains the primary operating point because q90 was predeclared before "
        "policy-validation in v1. The switch from marginal to class-conditional conformal "
        "changes the uncertainty set construction, not the predeclared uncertainty-gate percentile."
    ),
    "sensitivity_policy_ids":["mondrian_v2_q80","mondrian_v2_q95"],
    "policy_validation_used_to_select_v2":False,
    "future_roles_read":False,
    "candidates":candidate_policies,
    "calibration_block_state_counts":state_counts,
}

family_path=OUT/"policy_family_v2.json"
signals_path=OUT/"calibration_policy_states_v2.csv.gz"
protocol_path=OUT/"protocol.json"

family_path.write_text(json.dumps(policy_family,indent=2)+"\n")
signals.to_csv(signals_path,index=False,compression={"method":"gzip","mtime":0})

protocol={
    "status":"MONDRIAN_V2_OPERATIONAL_POLICY_FROZEN",
    "scientific_acceptance":False,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "horizon_hours":72,
    "source_role":ROLE,
    "rows":EXPECTED_ROWS,
    "probability_calibration":{"sharp":"raw_identity","aia":"raw_identity","sharp_aia":"raw_identity"},
    "conformal_method":"class_conditional_mondrian_split_conformal",
    "primary_alpha":PRIMARY_ALPHA,
    "primary_policy_id":"mondrian_v2_q90",
    "sensitivity_policy_ids":["mondrian_v2_q80","mondrian_v2_q95"],
    "candidate_cutoffs_reused_from_v1_freeze":True,
    "policy_validation_used_to_fit_or_select_v2":False,
    "retrospective_cycle25_outcomes_read":False,
    "supplementary_2026_outcomes_read":False,
    "next_stage":(
        "Evaluate the frozen primary and sensitivity v2 policies once on still-unopened "
        "retrospective_cycle25, then on supplementary_2026, reporting availability, "
        "flare retention, false alarms, abstained flares, issued-case skill, and class-conditional coverage."
    ),
    "source_sha256":{
        "master_predictions":MASTER_SHA,
        "sharp_predictions":SHARP_SHA,
        "mondrian_protocol":sha256(MOND_PROTOCOL),
        "mondrian_thresholds":sha256(MOND_THRESHOLDS),
        "candidate_cutoffs":sha256(CUTS),
        "alarm_protocol":sha256(ALARM_PROTOCOL),
        "primary_policy_declaration":sha256(PRIMARY_DECL),
    },
    "output_sha256":{
        family_path.name:sha256(family_path),
        signals_path.name:sha256(signals_path),
    },
}
protocol_path.write_text(json.dumps(protocol,indent=2)+"\n")

print("===== MONDRIAN V2 POLICY FREEZE COMPLETE =====")
print("rows:",EXPECTED_ROWS)
print("policy-validation used to fit/select v2: False")
print("Cycle 25 outcomes read: False")
print("supplementary 2026 outcomes read: False")
print("primary policy: mondrian_v2_q90")
print("\nCalibration-block state counts:")
print(json.dumps(state_counts,indent=2))
print("\nProtocol:")
print(json.dumps(protocol,indent=2))
print("\nSaved:",protocol_path)
print("Saved:",family_path)
print("Saved:",signals_path)


===== MONDRIAN V2 POLICY FREEZE COMPLETE =====
rows: 4168
policy-validation used to fit/select v2: False
Cycle 25 outcomes read: False
supplementary 2026 outcomes read: False
primary policy: mondrian_v2_q90

Calibration-block state counts:
{
  "mondrian_v2_q80": {
    "NORMAL": 2819,
    "DEGRADED": 617,
    "ABSTAIN": 732,
    "availability_rate": 0.8243761996161229
  },
  "mondrian_v2_q90": {
    "NORMAL": 3064,
    "DEGRADED": 689,
    "ABSTAIN": 415,
    "availability_rate": 0.9004318618042226
  },
  "mondrian_v2_q95": {
    "NORMAL": 3242,
    "DEGRADED": 738,
    "ABSTAIN": 188,
    "availability_rate": 0.95489443378119
  }
}

Protocol:
{
  "status": "MONDRIAN_V2_OPERATIONAL_POLICY_FROZEN",
  "scientific_acceptance": false,
  "created_utc": "2026-10-05T17:30:23.318707+00:00",
  "horizon_hours": 72,
  "source_role": "conformal_calibration",
  "rows": 4168,
  "probability_calibration": {
    "sharp": "raw_identity",
    "aia": "raw_identity",
    "sharp_aia": "raw_identity"
  },
  